# 07 — Hourly linear regression

24 linear regressions, one per local hour, built **one change at a time**: each change is kept only if it is significant in a Diebold-Mariano test and makes physical sense.

- **Input:** dataset (05) and naive forecasts (06)
- **Output:** `data/processed/linear_validation_forecasts.parquet`, `results/07_linear_*`
- **Chosen model:** `linear_v3_avg` — validation MAE 9.32 €/MWh, rMAE 0.76.
- **Key finding:** a *gas × residual load* interaction cuts the error by 14%: the slope of the merit order grows with the cost of fuel. Averaging an expanding and a 365-day rolling window adds another significant gain (forecast combination).

In [1]:
# --- Make src/ importable and reload it automatically when it changes ---------
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import forecasting as fc

TARGET = fc.TARGET

# --- Load dataset and naive forecasts ------------------------------------------
dataset = pd.read_parquet(fc.PROCESSED / "dataset_hourly_2021_2025q3.parquet")
naive = pd.read_parquet(fc.PROCESSED / "naive_forecasts_2021_2025q3.parquet")

# Only the forecast columns come from the naive file: split, period and
# calendar columns are rebuilt from src/, so there is one single definition
NAIVE_COLUMNS = ["naive_d1", "naive_d7", "naive_mean7", "naive_epf"]
dataset = dataset.merge(naive[["ts_utc"] + NAIVE_COLUMNS], on="ts_utc",
                        how="left", validate="one_to_one")

dataset = fc.add_split_columns(dataset)
dataset = fc.add_calendar_features(dataset)

# --- Checks ----------------------------------------------------------------------
assert len(dataset) == 41615
assert dataset.loc[dataset["split"] != "train", NAIVE_COLUMNS].notna().all().all()

# Holidays found in the perimeter: a quick look to see that the list makes sense
holidays_found = (
    dataset.loc[dataset["is_holiday"], "local_date"].drop_duplicates()
           .dt.strftime("%Y-%m-%d (%a)")
)
print(f"{len(holidays_found)} holiday days in the perimeter; first year:")
print(holidays_found.head(12).to_string(index=False))
print()
print(dataset[["ts_local", "local_hour", "day_of_week", "month", "is_holiday", "split"]]
      .head(3).to_string(index=False))

56 holiday days in the perimeter; first year:
2021-01-01 (Fri)
2021-01-06 (Wed)
2021-04-04 (Sun)
2021-04-05 (Mon)
2021-04-25 (Sun)
2021-05-01 (Sat)
2021-06-02 (Wed)
2021-08-15 (Sun)
2021-11-01 (Mon)
2021-12-08 (Wed)
2021-12-25 (Sat)
2021-12-26 (Sun)

                 ts_local  local_hour  day_of_week  month  is_holiday split
2021-01-01 00:00:00+01:00           0            4      1        True train
2021-01-01 01:00:00+01:00           1            4      1        True train
2021-01-01 02:00:00+01:00           2            4      1        True train


In [2]:
# --- Linear model v1: 24 hourly regressions --------------------------------------
from sklearn.linear_model import LinearRegression

LINEAR_FEATURES = [
    "naive_d1",                 # price, same hour yesterday (known at 12:00 D-1)
    "naive_d7",                 # price, same hour one week ago
    "residual_load_mw",         # NOT its three components: perfect collinearity
    "gas_price_d1_eur_mwh",
    "is_holiday",
]


def make_design_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Numeric feature matrix for the linear model.

    Day-of-week dummies are built by hand with FIXED columns (Tuesday..Sunday,
    Monday is the baseline absorbed by the intercept). pd.get_dummies would
    only create the columns for the days present in df: on a single
    prediction day it would produce 1 column instead of 6 and break predict.
    """
    X = df[LINEAR_FEATURES].astype(float)
    for day in range(1, 7):
        X[f"dow_{day}"] = (df["day_of_week"] == day).astype(float)
    return X


def fit_hourly_linear(train: pd.DataFrame) -> dict:
    """One LinearRegression per local hour: 04:00 and 19:00 have different dynamics."""
    models = {}
    for hour, rows in train.groupby("local_hour"):
        # The first week of 2021 has no naive_d7 yet: drop incomplete rows
        rows = rows.dropna(subset=LINEAR_FEATURES)
        models[hour] = LinearRegression().fit(make_design_matrix(rows), rows[TARGET])
    return models


def predict_hourly_linear(models: dict, day: pd.DataFrame) -> np.ndarray:
    """Forecast every row of one delivery day with the model of its hour."""
    forecast = pd.Series(np.nan, index=day.index)
    for hour, rows in day.groupby("local_hour"):
        forecast.loc[rows.index] = models[hour].predict(make_design_matrix(rows))
    return forecast.to_numpy()


# --- Walk-forward on the validation year: daily refit, expanding window ----------
import time

start = time.perf_counter()
dataset["linear_v1"] = fc.walk_forward(
    dataset, fit_hourly_linear, predict_hourly_linear, split="validation",
)
print(f"walk-forward done in {time.perf_counter() - start:.0f} s")

# --- Evaluate against the naive forecasts (validation only) ------------------------
results_v1 = fc.evaluate(dataset, NAIVE_COLUMNS + ["linear_v1"], split="validation")
print(results_v1[results_v1["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "WAPE_pct", "rMAE"]].round(2).to_string())

walk-forward done in 28 s
               MAE   RMSE  WAPE_pct  rMAE
forecast                                 
naive_d1     12.58  18.86     11.69  1.02
naive_d7     14.42  20.94     13.40  1.17
naive_mean7  11.89  17.15     11.05  0.97
naive_epf    12.30  18.35     11.43  1.00
linear_v1    11.30  15.12     10.50  0.92


In [3]:
# --- What did the model learn? Coefficients of the first validation-day fit ------
# Fitting on the train split gives exactly the model used for 1 Oct 2023
train = dataset.loc[dataset["split"] == "train"]
models = fit_hourly_linear(train)

feature_names = make_design_matrix(train.head(1)).columns
coefficients = pd.DataFrame(
    {hour: model.coef_ for hour, model in models.items()},
    index=feature_names,
).T
coefficients["intercept"] = [model.intercept_ for model in models.values()]

# EUR/MWh per MW is unreadable: express the residual-load effect per GW
coefficients["residual_load_eur_per_gw"] = coefficients.pop("residual_load_mw") * 1000

SHOW_HOURS = [4, 8, 13, 19]
print(coefficients.loc[SHOW_HOURS].T.round(2).to_string())

                             4      8      13     19
naive_d1                   0.41   0.31   0.34   0.53
naive_d7                   0.01   0.13   0.12   0.10
gas_price_d1_eur_mwh       1.07   1.34   1.07   0.99
is_holiday                 6.74  -9.43   7.95  -1.94
dow_1                      1.24 -18.89 -21.99 -12.74
dow_2                     -4.66 -23.11 -26.21 -21.13
dow_3                     -1.35 -24.80 -23.57 -21.68
dow_4                     -4.38 -24.20 -27.42 -29.80
dow_5                      1.61 -48.28 -31.75 -30.41
dow_6                      3.79 -39.85 -22.97 -22.61
intercept                -50.83 -15.37 -42.61 -17.56
residual_load_eur_per_gw   2.95   2.05   3.06   1.51


In [4]:
# --- Is linear_v1 significantly better than each naive forecast? -------------------
# Diebold-Mariano test
dm_rows = [
    fc.diebold_mariano(dataset, naive_col, "linear_v1", split="validation", loss=loss)
    for naive_col in NAIVE_COLUMNS
    for loss in ["abs", "squared"]
]
dm_table = pd.DataFrame(dm_rows)
print(dm_table.drop(columns=["forecast_b", "n_days"]).round(4).to_string(index=False))

 forecast_a    loss  mean_daily_diff  dm_stat  p_value_b_better
   naive_d1     abs           1.2812   3.0381            0.0012
   naive_d1 squared         126.9453   5.0303            0.0000
   naive_d7     abs           3.1223   6.5888            0.0000
   naive_d7 squared         209.9340   7.3714            0.0000
naive_mean7     abs           0.5861   1.4353            0.0756
naive_mean7 squared          65.3813   3.2182            0.0006
  naive_epf     abs           0.9957   2.5643            0.0052
  naive_epf squared         108.3019   4.5752            0.0000


In [10]:
# --- A reusable hourly linear model: feature list and price transform as parameters ---
PRICE_FEATURES = ["naive_d1", "naive_d7"]   # features on the same scale as the target


def make_hourly_linear(features: list, asinh_prices: bool = False):
    """Return (fit, predict, design) for 24 hourly regressions on `features`.

    asinh_prices=True applies the variance-stabilising transform used by LEAR to
    the target AND to the lagged prices:
        z = asinh((price - median) / MAD)
    median and MAD are computed on each training window, so no information from
    the forecast day is used. Predictions are transformed back with sinh.
    """

    def design(df: pd.DataFrame, scale: dict | None = None) -> pd.DataFrame:
        X = df[features].astype(float)
        if scale is not None:
            for col in PRICE_FEATURES:
                if col in X:
                    X[col] = np.arcsinh((X[col] - scale["median"]) / scale["mad"])
        # Fixed day-of-week columns (Tuesday..Sunday), Monday = reference category
        for day in range(1, 7):
            X[f"dow_{day}"] = (df["day_of_week"] == day).astype(float)
        return X

    def fit(train: pd.DataFrame) -> dict:
        scale = None
        if asinh_prices:
            median = train[TARGET].median()
            mad = (train[TARGET] - median).abs().median()   # median absolute deviation
            scale = {"median": median, "mad": mad}

        models = {"scale": scale}
        for hour, rows in train.groupby("local_hour"):
            rows = rows.dropna(subset=features)
            y = rows[TARGET]
            if scale is not None:
                y = np.arcsinh((y - scale["median"]) / scale["mad"])
            models[hour] = LinearRegression().fit(design(rows, scale), y)
        return models

    def predict(models: dict, day: pd.DataFrame) -> np.ndarray:
        scale = models["scale"]
        forecast = pd.Series(np.nan, index=day.index)
        for hour, rows in day.groupby("local_hour"):
            z = models[hour].predict(design(rows, scale))
            if scale is not None:
                z = np.sinh(z) * scale["mad"] + scale["median"]   # back to EUR/MWh
            forecast.loc[rows.index] = z
        return forecast.to_numpy()

    return fit, predict, design


# --- Regression check: without the transform, the factory must reproduce linear_v1 ---
train = dataset.loc[dataset["split"] == "train"]
fit_v1, predict_v1, design_v1 = make_hourly_linear(LINEAR_FEATURES)
old_models, new_models = fit_hourly_linear(train), fit_v1(train)
assert all(np.allclose(old_models[h].coef_, new_models[h].coef_) for h in range(24))
print("factory reproduces linear_v1: OK")

factory reproduces linear_v1: OK


In [6]:
# --- Modification A: interaction gas x residual load ----------------------------------
# The slope of the merit order scales with fuel cost: one extra GW of residual load
# costs more when gas is expensive. A linear model cannot learn this by itself,
# so the product of the two becomes a new column.
# Residual load in GW here, so the coefficient stays readable:
# EUR/MWh of power, per GW of residual load, per EUR/MWh of gas.
dataset["gas_x_residual_load"] = (
    dataset["gas_price_d1_eur_mwh"] * dataset["residual_load_mw"] / 1000
)

FEATURES_V2 = LINEAR_FEATURES + ["gas_x_residual_load"]
fit_v2, predict_v2, design_v2 = make_hourly_linear(FEATURES_V2)

start = time.perf_counter()
dataset["linear_v2"] = fc.walk_forward(dataset, fit_v2, predict_v2, split="validation")
print(f"walk-forward done in {time.perf_counter() - start:.0f} s\n")

# --- Metrics: v1 vs v2 ------------------------------------------------------------------
results_v2 = fc.evaluate(dataset, NAIVE_COLUMNS + ["linear_v1", "linear_v2"], split="validation")
print(results_v2[results_v2["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "WAPE_pct", "rMAE"]].round(2).to_string())

# --- Is v2 significantly better than v1? ----------------------------------------------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, "linear_v1", "linear_v2", split="validation", loss=loss)
    for loss in ["abs", "squared"]
]).drop(columns=["n_days"]).round(4).to_string(index=False))

# --- Physics check: does the residual-load slope grow with gas? ------------------------
# Slope of price w.r.t. residual load = b_RL (per GW) + b_interaction * gas
# Re-select the train split: the `train` of the previous cell was taken BEFORE
# the interaction column existed
train = dataset.loc[dataset["split"] == "train"]
models_v2 = fit_v2(train)
columns_v2 = list(design_v2(train.head(1)).columns)
i_rl, i_int = columns_v2.index("residual_load_mw"), columns_v2.index("gas_x_residual_load")

slopes = pd.DataFrame({
    f"gas {gas} EUR/MWh": {
        hour: models_v2[hour].coef_[i_rl] * 1000 + models_v2[hour].coef_[i_int] * gas
        for hour in [4, 8, 13, 19]
    }
    for gas in [30, 100]
})
print("\nEUR/MWh per extra GW of residual load, by hour:")
print(slopes.round(2).to_string())

walk-forward done in 29 s

               MAE   RMSE  WAPE_pct  rMAE
forecast                                 
naive_d1     12.58  18.86     11.69  1.02
naive_d7     14.42  20.94     13.40  1.17
naive_mean7  11.89  17.15     11.05  0.97
naive_epf    12.30  18.35     11.43  1.00
linear_v1    11.30  15.12     10.50  0.92
linear_v2     9.74  13.62      9.05  0.79

forecast_a forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
 linear_v1  linear_v2     abs           1.5620   9.6208               0.0
 linear_v1  linear_v2 squared          42.9928   5.6757               0.0

EUR/MWh per extra GW of residual load, by hour:
    gas 30 EUR/MWh  gas 100 EUR/MWh
4             2.01             3.41
8             0.52             3.07
13            1.68             5.07
19            0.59             2.23


In [8]:
# --- Modification B: was yesterday a holiday? -----------------------------------------
# Rebuild the calendar columns: add_calendar_features now also creates
# `yesterday_holiday` (autoreload has already picked up the new module)
dataset = fc.add_calendar_features(dataset)

validation = dataset.loc[dataset["split"] == "validation"]
n_days = validation.loc[validation["yesterday_holiday"], "local_date"].nunique()
print(f"days after a holiday in validation: {n_days}")

FEATURES_V3 = FEATURES_V2 + ["yesterday_holiday"]
fit_v3, predict_v3, design_v3 = make_hourly_linear(FEATURES_V3)

start = time.perf_counter()
dataset["linear_v3"] = fc.walk_forward(dataset, fit_v3, predict_v3, split="validation")
print(f"walk-forward done in {time.perf_counter() - start:.0f} s\n")

# --- Metrics: v2 vs v3, overall ----------------------------------------------------------
results_v3 = fc.evaluate(dataset, ["linear_v1", "linear_v2", "linear_v3"], split="validation")
print(results_v3[results_v3["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "rMAE"]].round(2).to_string())

# --- ...and only on the days the feature is about ---------------------------------------
validation = dataset.loc[dataset["split"] == "validation"]
after_holiday = validation["yesterday_holiday"]
print("\nMAE on days after a holiday vs all other days:")
print(pd.DataFrame({
    col: {
        "after holiday": (validation[TARGET] - validation[col])[after_holiday].abs().mean(),
        "other days": (validation[TARGET] - validation[col])[~after_holiday].abs().mean(),
    }
    for col in ["linear_v2", "linear_v3"]
}).round(2).to_string())

# --- Significance over the whole validation year -----------------------------------------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, "linear_v2", "linear_v3", split="validation", loss=loss)
    for loss in ["abs", "squared"]
]).drop(columns=["n_days"]).round(4).to_string(index=False))

days after a holiday in validation: 12
walk-forward done in 29 s

             MAE   RMSE  rMAE
forecast                     
linear_v1  11.30  15.12  0.92
linear_v2   9.74  13.62  0.79
linear_v3   9.66  13.53  0.79

MAE on days after a holiday vs all other days:
               linear_v2  linear_v3
after holiday      16.18      13.56
other days          9.52       9.53

forecast_a forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
 linear_v2  linear_v3     abs           0.0796   1.3079            0.0955
 linear_v2  linear_v3 squared           2.5837   0.7952            0.2132


In [9]:
# --- Modification D: rolling training window --------------------------------------
# Same features as v3; only the training window changes. With window_days=365 the
# model forgets everything older than one year: by 1 Oct 2023 the peak of the gas
# crisis (summer 2022) has already left the training set.
start = time.perf_counter()
dataset["linear_v3_roll365"] = fc.walk_forward(
    dataset, fit_v3, predict_v3, split="validation", window_days=365,
)
print(f"walk-forward done in {time.perf_counter() - start:.0f} s\n")

# Calibration window averaging: the mean of the expanding and the rolling forecasts.
# No new fit needed. Short windows react to regime changes, long ones are stable.
dataset["linear_v3_avg"] = dataset[["linear_v3", "linear_v3_roll365"]].mean(axis=1)

CANDIDATES = ["linear_v3", "linear_v3_roll365", "linear_v3_avg"]
results_d = fc.evaluate(dataset, ["naive_epf"] + CANDIDATES, split="validation")
print(results_d[results_d["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "rMAE"]].round(2).to_string())

# --- Significance against the expanding-window v3 ------------------------------------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, "linear_v3", candidate, split="validation", loss=loss)
    for candidate in ["linear_v3_roll365", "linear_v3_avg"]
    for loss in ["abs", "squared"]
]).drop(columns=["forecast_a", "n_days"]).round(4).to_string(index=False))

walk-forward done in 26 s

                     MAE   RMSE  rMAE
forecast                             
naive_epf          12.30  18.35  1.00
linear_v3           9.66  13.53  0.79
linear_v3_roll365   9.51  13.23  0.77
linear_v3_avg       9.32  13.08  0.76

       forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
linear_v3_roll365     abs           0.1490   1.0431            0.1484
linear_v3_roll365 squared           7.9183   1.5657            0.0587
    linear_v3_avg     abs           0.3432   4.7327            0.0000
    linear_v3_avg squared          11.8339   4.5658            0.0000


In [11]:
# --- Modification C: asinh transform of prices -------------------------------------
# Rebuild the v3 functions with the new factory (same features, no transform),
# then the same model with the asinh transform.
fit_v3, predict_v3, design_v3 = make_hourly_linear(FEATURES_V3)
fit_v4, predict_v4, design_v4 = make_hourly_linear(FEATURES_V3, asinh_prices=True)

# Same window setup as the current best model: expanding, rolling 365, and their mean
start = time.perf_counter()
dataset["linear_v4"] = fc.walk_forward(dataset, fit_v4, predict_v4, split="validation")
dataset["linear_v4_roll365"] = fc.walk_forward(
    dataset, fit_v4, predict_v4, split="validation", window_days=365,
)
dataset["linear_v4_avg"] = dataset[["linear_v4", "linear_v4_roll365"]].mean(axis=1)
print(f"walk-forward done in {time.perf_counter() - start:.0f} s\n")

CANDIDATES = ["linear_v3", "linear_v3_roll365", "linear_v3_avg",
              "linear_v4", "linear_v4_roll365", "linear_v4_avg"]
results_c = fc.evaluate(dataset, ["naive_epf"] + CANDIDATES, split="validation")
print(results_c[results_c["period"] == "all"]
      .set_index("forecast")[["MAE", "RMSE", "rMAE"]].round(2).to_string())

# --- Does asinh help? Compare each v4 variant with its v3 counterpart ----------------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, f"linear_v3{suffix}", f"linear_v4{suffix}",
                       split="validation", loss=loss)
    for suffix in ["", "_roll365", "_avg"]
    for loss in ["abs", "squared"]
]).drop(columns=["n_days"]).round(4).to_string(index=False))


walk-forward done in 64 s

                     MAE   RMSE  rMAE
forecast                             
naive_epf          12.30  18.35  1.00
linear_v3           9.66  13.53  0.79
linear_v3_roll365   9.51  13.23  0.77
linear_v3_avg       9.32  13.08  0.76
linear_v4           9.69  13.61  0.79
linear_v4_roll365   9.17  13.16  0.75
linear_v4_avg       9.15  12.94  0.74

       forecast_a        forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
        linear_v3         linear_v4     abs          -0.0271  -0.1478            0.5588
        linear_v3         linear_v4 squared          -2.1781  -0.3180            0.6248
linear_v3_roll365 linear_v4_roll365     abs           0.3385   3.1797            0.0007
linear_v3_roll365 linear_v4_roll365 squared           1.9293   0.4460            0.3278
    linear_v3_avg     linear_v4_avg     abs           0.1691   1.3806            0.0837
    linear_v3_avg     linear_v4_avg squared           3.7161   0.8523            0.1970


In [12]:
# --- Persist validation forecasts, metrics and significance tests ---------------------
LINEAR_VERSIONS = [
    "linear_v1", "linear_v2", "linear_v3",
    "linear_v3_roll365", "linear_v3_avg",
    "linear_v4", "linear_v4_roll365", "linear_v4_avg",
]
FINAL_LINEAR = "linear_v3_avg"   # chosen on validation; the test set is evaluated later, once

fc.RESULTS.mkdir(exist_ok=True)

# Validation forecasts of every version: the final comparison notebook and the
# XGBoost/LEAR notebooks can load them instead of re-running the walk-forwards
validation = dataset.loc[dataset["split"] == "validation"]
KEEP = ["ts_utc", "ts_local", "local_date", "local_hour", "period", "split", TARGET]
forecasts_out = validation[KEEP + NAIVE_COLUMNS + LINEAR_VERSIONS].reset_index(drop=True)

forecasts_path = fc.PROCESSED / "linear_validation_forecasts.parquet"
forecasts_out.to_parquet(forecasts_path, index=False)
pd.testing.assert_frame_equal(pd.read_parquet(forecasts_path), forecasts_out)

# Metrics of all versions against the naive forecasts
metrics = fc.evaluate(dataset, NAIVE_COLUMNS + LINEAR_VERSIONS, split="validation")
metrics.to_csv(fc.RESULTS / "07_linear_validation_metrics.csv", index=False)

# Each step tested against the model it modified: the path that led to the final choice
STEPS = [
    ("naive_epf", "linear_v1"),
    ("naive_mean7", "linear_v1"),
    ("linear_v1", "linear_v2"),                  # A: gas x residual load
    ("linear_v2", "linear_v3"),                  # B: yesterday_holiday
    ("linear_v3", "linear_v3_roll365"),          # D: rolling window
    ("linear_v3", "linear_v3_avg"),              # D: window averaging
    ("linear_v3", "linear_v4"),                  # C: asinh, expanding
    ("linear_v3_roll365", "linear_v4_roll365"),  # C: asinh, rolling
    ("linear_v3_avg", "linear_v4_avg"),          # C: asinh, averaged
]
dm_steps = pd.DataFrame([
    fc.diebold_mariano(dataset, a, b, split="validation", loss=loss)
    for a, b in STEPS
    for loss in ["abs", "squared"]
])
dm_steps.to_csv(fc.RESULTS / "07_linear_dm_tests.csv", index=False)

print(f"saved {forecasts_out.shape} -> {forecasts_path.name}")
print(f"saved {metrics.shape} -> results/07_linear_validation_metrics.csv")
print(f"saved {dm_steps.shape} -> results/07_linear_dm_tests.csv")
print(f"\nfinal linear model: {FINAL_LINEAR}")

saved (8784, 19) -> linear_validation_forecasts.parquet
saved (24, 8) -> results/07_linear_validation_metrics.csv
saved (18, 7) -> results/07_linear_dm_tests.csv

final linear model: linear_v3_avg


## Summary

**Model.** 24 linear regressions, one per local hour, refitted every day in a walk-forward over the validation year (1 Oct 2023 → 30 Sep 2024, 366 days). Day-of-week dummies with fixed columns (Monday = reference). All choices below were made on validation only; **the test set has not been looked at**.

| Version | Change | MAE | RMSE | rMAE | DM (abs) vs previous |
|---|---|---|---|---|---|
| `linear_v1` | lagged prices (D-1, D-7), residual load, gas D-1, holiday, day of week | 11.30 | 15.12 | 0.92 | vs EPF naive 2.56 (p 0.005) |
| `linear_v2` | **A** + gas × residual load | 9.74 | 13.62 | 0.79 | 9.62 (p ≈ 0) |
| `linear_v3` | **B** + yesterday was a holiday | 9.66 | 13.53 | 0.79 | 1.31 (p 0.10) |
| `linear_v3_roll365` | **D** rolling 365-day window | 9.51 | 13.23 | 0.77 | 1.04 (p 0.15) |
| **`linear_v3_avg`** | **D** mean of expanding and rolling | **9.32** | **13.08** | **0.76** | **4.73 (p ≈ 0)** |
| `linear_v4_avg` | **C** asinh transform, averaged windows | 9.15 | 12.94 | 0.74 | 1.38 vs v3_avg (p 0.08) |

**Final choice: `linear_v3_avg`** (rMAE 0.76, i.e. 24% better than the EPF naive).

**What each step taught:**
- **v1 coefficients make physical sense.** The gas coefficient (≈1) is conditional on yesterday's price; the long-run pass-through c / (1 − b₁ − b₇) is ≈ 1.8–2.7, the heat rate of a combined-cycle plant. Day-of-week dummies are corrections relative to yesterday, close to zero at night. A large coefficient does not mean an important feature: what matters is coefficient × typical variation.
- **A — interaction.** The biggest gain (−14% MAE). The residual-load slope grows with gas price at every hour (e.g. 13:00: 1.7 → 5.1 EUR/MWh per GW with gas at 30 vs 100): the merit-order slope scales with fuel cost. Without it, one slope had to fit both 2021 and the 2022 crisis.
- **B — yesterday_holiday.** −16% MAE on the 12 days after a holiday, no change elsewhere; not significant over the year because the cases are too few. Kept as a declared exception: motivated a priori (same mechanism as Monday copying Sunday) and costless.
- **D — windows.** The rolling window alone is not significant; the average of the two windows is. Forecast combination: the two windows make errors on different days.
- **C — asinh.** No effect with the expanding window (median/MAD are computed on a window that contains the crisis, so the transform stays almost linear); significant with the rolling window on MAE but not on RMSE. Rejected for the linear model: the gain over `linear_v3_avg` is not significant and coefficients lose their physical reading. It will be central in LEAR.

**Method note.** About ten comparisons were run on validation; validation metrics of the chosen model are therefore optimistic. The test year will be evaluated once, in a final notebook comparing the chosen linear, XGBoost and LEAR configurations.